# 🎬 h3出片工作台（MiniMax-H3 `ref2va` · Colab Pro+ H200 141GB 无损 BF16）

已预填当前工程 **`xiaoxing_wenhao_ep01` / `ep01` / `ONLY_PARTS = 'S01'`**，切到 **H200 (141GB)** 后点击菜单 **「代码执行程序 → 全部运行」** 即可一键出片：
- **默认无损配置**：`h200-max`（纯 BF16 `pruned` DiT 40.2GB + BF16 Qwen3-VL-32B 51.5GB + FP16 Video VAE + FP32 Audio VAE + `ref_image_size=max`），零画质折损。
- **秒拉云盘缓存**：自动从 `/content/drive/MyDrive/models/minimax_h3` 流式拉取已预热的 97.5 GB 权重（全程实时打印进度），免公网下载。
- **续跑 S02–S12 极简切法**：S01 验片满意后，只需把 `ONLY_PARTS` 改为空字符串 `''`（保持 `SKIP_EXISTING = True`），再点「全部运行」即可自动跳过 S01 连跑 S02–S12，且每出完 1 段立即写回 Google Drive 防丢。
- **防烧 CU 铁律**：`RUN_MODE = 'batch'` 跑完并刷盘回写 Google Drive 后，自动等待 10 秒落盘再执行 `runtime.unassign()` 销毁 GPU 停止计费。


In [ ]:
#@title 1. 硬件检查 + 挂载 Google Drive + 任务配置
import os, subprocess
from pathlib import Path
from google.colab import drive

print('=== GPU 与内存/磁盘检查 ===')
for cmd in (
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    ['free', '-h'],
    ['df', '-h', '/content'],
):
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout.strip())
drive.mount('/content/drive')

# ===== 任务配置（已预填 EP01 首段 S01）=====
SERIES_DRIVE_DIR = '/content/drive/MyDrive/ai_anime/xiaoxing_wenhao_ep01'  #@param {type:'string'}
EPISODE = 'ep01'  #@param {type:'string'}
ONLY_PARTS = 'S01'  #@param {type:'string'}  # 只跑 S01；留空 '' 则跑全部段（配合 SKIP_EXISTING 自动跳过已出段）
SKIP_EXISTING = True  #@param {type:'boolean'}  # 已存在的成片自动跳过（断点续跑）
USE_EXISTING_MANIFEST = True  #@param {type:'boolean'}  # 沿用本地编译好的 12 段 H3 manifest.json
PROFILE = 'h200-max'  #@param ['h200-max', 'bf16-raw', 'h200-fast', 'g4-balanced']
RESOLUTION = '768p'  #@param ['768p', '1080p', '480p']
REF_IMAGE_SIZE = 'max'  #@param ['max', 'match']
ENABLE_TURBO_4STEP = False  #@param {type:'boolean'}
RUN_MODE = 'batch'  #@param ['batch', 'webui']
MODEL_CACHE_DRIVE_DIR = '/content/drive/MyDrive/models/minimax_h3'  #@param {type:'string'}
HF_TOKEN = ''  #@param {type:'string'}  # 可选（公开仓库且已命中 G 盘缓存，留空即可）
AUTO_UNASSIGN_AFTER_BATCH = True  #@param {type:'boolean'}

assert Path(SERIES_DRIVE_DIR).exists(), f'云盘系列目录不存在: {SERIES_DRIVE_DIR}'
if MODEL_CACHE_DRIVE_DIR.strip():
    assert Path(MODEL_CACHE_DRIVE_DIR).exists(), f'云盘权重缓存目录不存在: {MODEL_CACHE_DRIVE_DIR}'
if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN.strip()
print(f'✅ 已就绪: {SERIES_DRIVE_DIR} / {EPISODE} | ONLY_PARTS={ONLY_PARTS or "ALL"} | SKIP_EXISTING={SKIP_EXISTING} | profile={PROFILE} | res={RESOLUTION}')


In [ ]:
#@title 2. 写入内置脚本并拷贝当前系列到本地高速 SSD
from pathlib import Path
import shutil

Path('/content/make_h3_jobs.py').write_text('# -*- coding: utf-8 -*-\n"""h3出片核心编译器：将 ai-anime-series (series.json + episode.json) 转为 MiniMax-H3 (ref2va) 任务包与 ComfyUI API 图。\n\n默认无损满血配置：\n  - --profile h200-max (BF16 pruned DiT 40.2GB + BF16 Qwen3-VL-32B 51.5GB + FP16 Video VAE + FP32 Audio VAE, ref_image_size=max)\n  - --resolution 768p (16:9 -> 1344x768, 9:16 -> 768x1344, 24fps)\n  - Colab G4 (96GB VRAM + 176GB RAM) 与 H200 (141GB) 均可零画质折损直跑。\n\n用法：\n  python make_h3_jobs.py SERIES ep01\n  python make_h3_jobs.py SERIES ep01 --profile h200-max --resolution 768p\n  python make_h3_jobs.py SERIES ep01 --profile h200-fast --resolution 480p --turbo\n  python make_h3_jobs.py SERIES ep01 --use-custom-prompt\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nimport random\nimport re\nimport sys\nfrom pathlib import Path\nfrom typing import Any\n\nfor _s in (sys.stdout, sys.stderr):\n    if hasattr(_s, "reconfigure"):\n        _s.reconfigure(encoding="utf-8")\n\nFPS = 24\nMAX_H3_REFS = 9\nMAX_H3_AUDIOS = 3\nMAX_H3_SEG_SEC = 15.0  # MiniMax-H3 官方训练上限 ~15s (362 frames @ 24fps)\n\nGRIDS = {4: (2, 2), 6: (2, 3), 8: (2, 4), 9: (3, 3)}\nCN_NUM = {4: "四", 6: "六", 8: "八", 9: "九"}\n\n# 对应 Comfy-Org/MiniMax-H3 官方公开仓库精确路径与体积（GB）\nPROFILES: dict[str, dict[str, Any]] = {\n    "h200-max": {\n        "label": "无损满血 BF16（默认推荐，G4 96GB / H200 零画质折损）",\n        "unet": "minimax_h3_ref2va_pruned_bf16.safetensors",\n        "clip": "qwen3vl_32b_minimax_h3_bf16.safetensors",\n        "video_vae": "minimax_h3_video_vae_fp16.safetensors",\n        "audio_vae": "minimax_h3_audio_vae_fp32.safetensors",\n        "ref_image_size": "max",\n        "steps": 20,\n        "scheduler": "beta",\n        "sampler": "res_multistep",\n        "approx_gb": 98,\n    },\n    "bf16-raw": {\n        "label": "原始未裁剪 BF16（极限对比档，总重 ~124GB，G4 / H200）",\n        "unet": "minimax_h3_ref2va_bf16.safetensors",\n        "clip": "qwen3vl_32b_minimax_h3_bf16.safetensors",\n        "video_vae": "minimax_h3_video_vae_fp16.safetensors",\n        "audio_vae": "minimax_h3_audio_vae_fp32.safetensors",\n        "ref_image_size": "max",\n        "steps": 20,\n        "scheduler": "beta",\n        "sampler": "res_multistep",\n        "approx_gb": 124,\n    },\n    "h200-fast": {\n        "label": "省 CU 高速档（FP8 Scaled + INT8 Qwen3-VL，~54GB）",\n        "unet": "minimax_h3_ref2va_pruned_fp8_scaled.safetensors",\n        "clip": "qwen3vl_32b_minimax_h3_int8_convrot.safetensors",\n        "video_vae": "minimax_h3_video_vae_fp16.safetensors",\n        "audio_vae": "minimax_h3_audio_vae_fp32.safetensors",\n        "ref_image_size": "match",\n        "steps": 20,\n        "scheduler": "beta",\n        "sampler": "res_multistep",\n        "approx_gb": 54,\n    },\n    "g4-balanced": {\n        "label": "超轻省内存档（INT8 ConvRot + NVFP4 Qwen3-VL，~43GB）",\n        "unet": "minimax_h3_ref2va_pruned_int8_convrot.safetensors",\n        "clip": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",\n        "video_vae": "minimax_h3_video_vae_fp16.safetensors",\n        "audio_vae": "minimax_h3_audio_vae_fp32.safetensors",\n        "ref_image_size": "match",\n        "steps": 20,\n        "scheduler": "beta",\n        "sampler": "res_multistep",\n        "approx_gb": 43,\n    },\n}\n\nTURBO_LORA = "minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors"\n\n# 分辨率严格为 32 的倍数，对齐官方 nodes_minimax_h3.py\nRESOLUTIONS: dict[str, dict[str, tuple[int, int]]] = {\n    "1080p": {"16:9": (1920, 1088), "9:16": (1088, 1920), "1:1": (1440, 1440), "4:3": (1440, 1088)},\n    "768p": {"16:9": (1344, 768), "9:16": (768, 1344), "1:1": (1024, 1024), "4:3": (1152, 864)},\n    "480p": {"16:9": (864, 480), "9:16": (480, 864), "1:1": (640, 640), "4:3": (736, 544)},\n}\n\n\ndef align_h3_frames(duration_sec: float) -> int:\n    """对齐 MiniMax-H3 官方 17k+5 帧网格（24fps，官方训练区间 124~362 帧）。"""\n    raw = max(5, round(duration_sec * FPS))\n    aligned = raw + (5 - (raw % 17)) % 17\n    if aligned < 124:\n        aligned = 124\n    if aligned > 362:\n        aligned = 362\n    return aligned\n\n\ndef fmt_sec(x: float) -> str:\n    return f"{x:.1f}".rstrip("0").rstrip(".")\n\n\ndef grid_of(n: int) -> tuple[int, int]:\n    if n in GRIDS:\n        return GRIDS[n]\n    cols = math.ceil(math.sqrt(n))\n    return math.ceil(n / cols), cols\n\n\ndef grid_name(n: int) -> str:\n    return f"{CN_NUM[n]}宫格" if n in CN_NUM else f"{n}格"\n\n\ndef validate_h3_prompt(prompt_text: str, num_images: int, num_audios: int) -> list[str]:\n    """硬校验 H3 提示词标记合法性，返回告警列表；致命错误直接抛 ValueError。"""\n    warnings: list[str] = []\n\n    legacy_tags = re.findall(r"@(?:图片?|音频|视频)\\s*\\d+", prompt_text)\n    if legacy_tags:\n        raise ValueError(\n            f"提示词中仍残留未转换的即梦语法标记 {sorted(set(legacy_tags))}，"\n            "请全部转换为 MiniMax-H3 原生标记 <Picture N> / <Audio M> 后再打包！"\n        )\n\n    pic_nums = [int(m) for m in re.findall(r"<Picture\\s+(\\d+)>", prompt_text)]\n    aud_nums = [int(m) for m in re.findall(r"<Audio\\s+(\\d+)>", prompt_text)]\n\n    bad_pics = sorted({n for n in pic_nums if n < 1 or n > num_images})\n    if bad_pics:\n        raise ValueError(f"提示词引用了越界的图片编号 <Picture {bad_pics}>，当前仅传入 {num_images} 张参考图！")\n\n    bad_auds = sorted({n for n in aud_nums if n < 1 or n > num_audios})\n    if bad_auds:\n        raise ValueError(f"提示词引用了越界的音频编号 <Audio {bad_auds}>，当前仅传入 {num_audios} 段参考音频！")\n\n    unused_pics = [i for i in range(1, num_images + 1) if i not in pic_nums]\n    if unused_pics:\n        warnings.append(f"有参考图未在提示词中被显式引用: {[f\'<Picture {i}>\' for i in unused_pics]}")\n    unused_auds = [i for i in range(1, num_audios + 1) if i not in aud_nums]\n    if unused_auds:\n        warnings.append(f"有参考音频未在提示词中被显式引用: {[f\'<Audio {i}>\' for i in unused_auds]}")\n\n    return warnings\n\n\ndef load_json(path: Path) -> dict[str, Any]:\n    with path.open("r", encoding="utf-8") as f:\n        return json.load(f)\n\n\ndef pick_items(items: list[dict], ids: list[str], label: str) -> list[dict]:\n    by_id = {x["id"]: x for x in items}\n    missing = [i for i in ids if i not in by_id]\n    if missing:\n        sys.exit(f"episode.json 中的{label} {missing} 在 series.json 中找不到")\n    return [by_id[i] for i in ids]\n\n\ndef resolve_board_path(ep_dir: Path, tag: str, multi: bool) -> Path:\n    candidates = [\n        f"storyboards/故事板_{tag}.png", f"storyboards/故事板_{tag}.jpg",\n        f"故事板_{tag}.png", f"故事板_{tag}.jpg", f"故事板_{tag}.jpeg", f"故事板_{tag}.webp",\n        f"storyboards/storyboard_{tag}.png", f"storyboard_{tag}.png", f"storyboard_{tag}.jpg",\n    ] if multi else [\n        "storyboards/故事板.png", "故事板.png", "故事板.jpg", "故事板.jpeg", "故事板.webp",\n        "storyboard.png", "storyboard.jpg",\n    ]\n    for name in candidates:\n        p = ep_dir / name\n        if p.exists():\n            return p\n    return ep_dir / candidates[0]\n\n\ndef collect_refs(series_dir: Path, ep_dir: Path, s: dict, e: dict, tag: str, multi: bool, part: dict | None = None):\n    p_cast = part["cast"] if (part and "cast" in part) else e.get("cast", [])\n    p_groups = part["groups"] if (part and "groups" in part) else e.get("groups", [])\n    p_scene = part["scene"] if (part and "scene" in part) else e.get("scene")\n    cast = pick_items(s.get("characters", []), p_cast, "角色")\n    groups = pick_items(s.get("groups", []), p_groups, "群体角色")\n    scenes = pick_items(s.get("scenes", []), [p_scene] if p_scene else [], "场景")\n\n    img_refs: list[dict[str, Any]] = []\n    audio_refs: list[dict[str, Any]] = []\n\n    for c in cast:\n        img_refs.append({"kind": "char", "name": c["name"], "path": series_dir / c["ref"], "item": c})\n        if c.get("voice_ref"):\n            audio_refs.append({"name": c["name"], "path": series_dir / c["voice_ref"]})\n    for g in groups:\n        if g.get("ref"):\n            img_refs.append({"kind": "group", "name": g["name"], "path": series_dir / g["ref"], "item": g})\n    for sc in scenes:\n        if sc.get("ref"):\n            img_refs.append({"kind": "scene", "name": sc["name"], "path": series_dir / sc["ref"], "item": sc})\n\n    board_path = resolve_board_path(ep_dir, tag, multi)\n    img_refs.append({"kind": "board", "name": "故事板", "path": board_path, "item": None})\n\n    if len(img_refs) > MAX_H3_REFS:\n        sys.exit(f"参考图数量 {len(img_refs)} 超过 MiniMax-H3 上限 {MAX_H3_REFS}，请合并配角设定图")\n    if len(audio_refs) > MAX_H3_AUDIOS:\n        sys.exit(f"参考音频数量 {len(audio_refs)} 超过 MiniMax-H3 上限 {MAX_H3_AUDIOS}")\n\n    for idx, r in enumerate(img_refs, 1):\n        r["idx"] = idx\n        r["tag"] = f"<Picture {idx}>"\n        r["exists"] = r["path"].exists()\n    for idx, a in enumerate(audio_refs, 1):\n        a["idx"] = idx\n        a["tag"] = f"<Audio {idx}>"\n        a["exists"] = a["path"].exists()\n\n    return img_refs, audio_refs, cast, groups, scenes\n\n\ndef build_h3_prompt(\n    s: dict,\n    e: dict,\n    shots: list[dict],\n    img_refs: list[dict],\n    audio_refs: list[dict],\n    groups: list[dict],\n    scenes: list[dict],\n) -> str:\n    board = next(r for r in img_refs if r["kind"] == "board")\n    sc_refs = [r for r in img_refs if r["kind"] == "scene"]\n    audio_by_name = {a["name"]: a["tag"] for a in audio_refs}\n\n    n = len(shots)\n    rows, cols = grid_of(n)\n    order_hint = "左上→右上→左下→右下" if (rows, cols) == (2, 2) else "从左到右、从上到下"\n\n    lines = [\n        f"根据 {board[\'tag\']}（{rows}行{cols}列{grid_name(n)}分镜故事板，按{order_hint}顺序推进）生成连贯全屏动画视频。",\n        f"严格要求：仅借鉴 {board[\'tag\']} 的景别、构图站位、动作节奏与镜头切换，严禁将分镜黑框、白边、宫格分割线或镜头序号渲染进画面。",\n        "",\n    ]\n\n    for r in img_refs:\n        if r["kind"] == "char":\n            v_hint = f" 说话音色严格参考 {audio_by_name[r[\'name\']]}。" if r["name"] in audio_by_name else ""\n            lines.append(f"{r[\'tag\']} 是角色「{r[\'name\']}」，外形、五官比例与配色必须与参考图严格一致：{r[\'item\'][\'look\']}。{v_hint}")\n        elif r["kind"] == "group":\n            lines.append(f"{r[\'tag\']} 是群体角色「{r[\'name\']}」的设定图：{r[\'item\'][\'look\']}。")\n    for g in groups:\n        if not g.get("ref"):\n            lines.append(f"群体角色「{g[\'name\']}」：{g[\'look\']}。")\n\n    place = scenes[0] if scenes else None\n    style_text = s.get("style", "").strip()\n    dialogue_mode = bool(s.get("dialogue", False))\n    if not dialogue_mode:\n        style_text += " 全片无台词，依靠夸张搞笑的肢体表演、表情和清脆卡通音效推动剧情。"\n    else:\n        style_text += " 中文清晰童声对白，说话口型与面部表情自然精准同步，配乐轻快童趣。"\n    if place:\n        style_text += f" 全片始终处于同一个{place[\'name\']}、同一个{e.get(\'time\', \'午后\')}：{place[\'look\']}。"\n        if sc_refs:\n            style_text += f" 场景风格与布景参考 {sc_refs[0][\'tag\']}。"\n    lines += ["", f"整体画风与场景：{style_text}", ""]\n\n    if e.get("continuity"):\n        lines += ["镜头连贯约束：" + "；".join(e["continuity"]) + "。", ""]\n\n    t = 0.0\n    for i, sh in enumerate(shots, 1):\n        dur = float(sh["dur"])\n        lines.append(f"镜头 {i}（{fmt_sec(t)}-{fmt_sec(t + dur)}秒）：{sh[\'desc\']}。")\n        if sh.get("dialogue"):\n            lines.append(f"对白（精准口型）：{sh[\'dialogue\']}")\n        if sh.get("sfx"):\n            lines.append(f"音效与配乐：{sh[\'sfx\']}")\n        lines.append("")\n        t += dur\n\n    lines.append("全程全屏24fps流畅渲染，画面干净无任何文字、字幕、水印、Logo或分镜边框。")\n    return "\\n".join(lines).strip() + "\\n"\n\n\ndef build_comfyui_api_graph(\n    prompt_text: str,\n    width: int,\n    height: int,\n    frame_length: int,\n    ref_image_names: list[str],\n    ref_audio_names: list[str],\n    profile: dict[str, Any],\n    ref_image_size: str,\n    use_turbo: bool,\n    seed: int,\n    output_prefix: str,\n) -> dict[str, Any]:\n    """构建源码级对齐 ComfyUI 官方 video_minimax_h3_r2v.json 的 /prompt API 图。"""\n    graph: dict[str, Any] = {\n        "1": {"class_type": "UNETLoader", "inputs": {"unet_name": profile["unet"], "weight_dtype": "default"}},\n        "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": profile["clip"], "type": "minimax", "device": "default"}},\n        "3": {"class_type": "VAELoader", "inputs": {"vae_name": profile["video_vae"]}},\n        "4": {"class_type": "VAELoader", "inputs": {"vae_name": profile["audio_vae"]}},\n    }\n\n    model_ref: list[Any] = ["1", 0]\n    steps = profile["steps"]\n    if use_turbo:\n        graph["5"] = {\n            "class_type": "LoraLoaderModelOnly",\n            "inputs": {"model": ["1", 0], "lora_name": TURBO_LORA, "strength_model": 1.0},\n        }\n        model_ref = ["5", 0]\n        steps = 4\n\n    h3_inputs: dict[str, Any] = {\n        "clip": ["2", 0],\n        "vae": ["3", 0],\n        "audio_vae": ["4", 0],\n        "prompt": prompt_text,\n        "width": width,\n        "height": height,\n        "length": frame_length,\n        "ref_image_size": ref_image_size,\n    }\n\n    next_id = 10\n    for idx, fname in enumerate(ref_image_names):\n        nid = str(next_id)\n        next_id += 1\n        graph[nid] = {"class_type": "LoadImage", "inputs": {"image": fname}}\n        h3_inputs[f"ref_images.ref_image_{idx}"] = [nid, 0]\n\n    for idx, fname in enumerate(ref_audio_names):\n        nid = str(next_id)\n        next_id += 1\n        graph[nid] = {"class_type": "LoadAudio", "inputs": {"audio": fname}}\n        h3_inputs[f"ref_audios.ref_audio_{idx}"] = [nid, 0]\n\n    graph["30"] = {"class_type": "MiniMaxH3ReferenceToVideo", "inputs": h3_inputs}\n    graph["31"] = {"class_type": "BasicGuider", "inputs": {"model": model_ref, "conditioning": ["30", 0]}}\n    graph["32"] = {"class_type": "KSamplerSelect", "inputs": {"sampler_name": profile["sampler"]}}\n    graph["33"] = {\n        "class_type": "BasicScheduler",\n        "inputs": {"model": model_ref, "scheduler": profile["scheduler"], "steps": steps, "denoise": 1.0},\n    }\n    graph["34"] = {"class_type": "RandomNoise", "inputs": {"noise_seed": seed}}\n    graph["35"] = {\n        "class_type": "SamplerCustomAdvanced",\n        "inputs": {\n            "noise": ["34", 0],\n            "guider": ["31", 0],\n            "sampler": ["32", 0],\n            "sigmas": ["33", 0],\n            "latent_image": ["30", 1],\n        },\n    }\n    graph["36"] = {"class_type": "VAEDecode", "inputs": {"samples": ["35", 0], "vae": ["3", 0]}}\n    graph["37"] = {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["35", 0], "vae": ["4", 0]}}\n    graph["38"] = {\n        "class_type": "CreateVideo",\n        "inputs": {"images": ["36", 0], "audio": ["37", 0], "fps": float(FPS)},\n    }\n    graph["39"] = {\n        "class_type": "SaveVideo",\n        "inputs": {"video": ["38", 0], "filename_prefix": output_prefix, "format": "mp4"},\n    }\n    return graph\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description="将 ai-anime-series 剧本转为 MiniMax-H3 (ref2va) 批处理任务")\n    parser.add_argument("series_dir", help="系列目录（含 series.json 和 refs/）")\n    parser.add_argument("episode", help="集目录名（如 ep01）")\n    parser.add_argument("--profile", choices=list(PROFILES.keys()), default="h200-max")\n    parser.add_argument("--resolution", choices=list(RESOLUTIONS.keys()), default="768p")\n    parser.add_argument("--ref-size", choices=["match", "max"], default=None, help="覆盖 profile 默认 ref_image_size")\n    parser.add_argument("--turbo", action="store_true", help="启用官方 4-step Turbo LoRA 快速抽卡")\n    parser.add_argument("--seed", type=int, default=None, help="固定随机种子（默认按段随机）")\n    parser.add_argument("--use-custom-prompt", action="store_true", help="若 epNN/h3_jobs/custom_h3_prompt*.txt 存在则优先采用精修提示词")\n    args = parser.parse_args()\n\n    series_dir = Path(args.series_dir).resolve()\n    ep_dir = series_dir / args.episode\n    s = load_json(series_dir / "series.json")\n    e = load_json(ep_dir / "episode.json")\n\n    parts = e.get("parts") or []\n    if not parts:\n        sys.exit("episode.json 中缺少 parts")\n\n    ratio = s.get("ratio", "16:9")\n    if ratio not in RESOLUTIONS[args.resolution]:\n        sys.exit(f"不支持的画幅比例 {ratio}，支持: {list(RESOLUTIONS[args.resolution].keys())}")\n    width, height = RESOLUTIONS[args.resolution][ratio]\n\n    profile = dict(PROFILES[args.profile])\n    ref_image_size = args.ref_size or profile["ref_image_size"]\n\n    out_dir = ep_dir / "h3_jobs"\n    out_dir.mkdir(parents=True, exist_ok=True)\n    multi = len(parts) > 1\n    jobs: list[dict[str, Any]] = []\n\n    for k, part in enumerate(parts):\n        tag = str(part.get("id") or (f"P{k + 1:02d}" if k >= 8 else "ABCDEFGH"[k]))\n        shots = part.get("shots") or []\n        seg_dur = sum(float(sh["dur"]) for sh in shots)\n        if seg_dur > MAX_H3_SEG_SEC + 0.5:\n            print(\n                f"⚠️ 警告：第 {tag} 段总时长 {fmt_sec(seg_dur)}s 超过 MiniMax-H3 原生 15s 上限，"\n                f"帧数将截断在 362 帧（15.08s）。建议将本段拆成多个 4 格（每段 10-15s）！",\n                file=sys.stderr,\n            )\n\n        frame_length = align_h3_frames(seg_dur)\n        img_refs, audio_refs, _cast, groups, scenes = collect_refs(series_dir, ep_dir, s, e, tag, multi, part=part)\n\n        part_ctx = dict(e)\n        for _k in ("time", "continuity", "scene"):\n            if _k in part and part[_k]:\n                part_ctx[_k] = part[_k]\n\n        part_suffix = f"_{tag}" if multi else ""\n        custom_path = out_dir / f"custom_h3_prompt{part_suffix}.txt"\n        if args.use_custom_prompt and custom_path.exists():\n            prompt_text = custom_path.read_text(encoding="utf-8").strip() + "\\n"\n            print(f"📝 段 {tag} 使用精修自定义 H3 提示词: {custom_path.name}")\n        else:\n            prompt_text = build_h3_prompt(s, part_ctx, shots, img_refs, audio_refs, groups, scenes)\n\n        warns = validate_h3_prompt(prompt_text, len(img_refs), len(audio_refs))\n        for w in warns:\n            print(f"⚠️ 段 {tag} 校验提醒: {w}", file=sys.stderr)\n\n        seed = (args.seed + k) if args.seed is not None else random.randint(1, 2**31 - 1)\n        staged_imgs = [f"h3_{args.episode}{part_suffix}_img{r[\'idx\']}{r[\'path\'].suffix or \'.png\'}" for r in img_refs]\n        staged_auds = [f"h3_{args.episode}{part_suffix}_aud{a[\'idx\']}{a[\'path\'].suffix or \'.wav\'}" for a in audio_refs]\n        output_prefix = f"h3_{args.episode}{part_suffix}"\n\n        api_graph = build_comfyui_api_graph(\n            prompt_text=prompt_text,\n            width=width,\n            height=height,\n            frame_length=frame_length,\n            ref_image_names=staged_imgs,\n            ref_audio_names=staged_auds,\n            profile=profile,\n            ref_image_size=ref_image_size,\n            use_turbo=args.turbo,\n            seed=seed,\n            output_prefix=output_prefix,\n        )\n\n        prompt_file = out_dir / f"h3_prompt{part_suffix}.txt"\n        api_file = out_dir / f"comfyui_api{part_suffix}.json"\n        prompt_file.write_text(prompt_text, encoding="utf-8")\n        api_file.write_text(json.dumps(api_graph, ensure_ascii=False, indent=2), encoding="utf-8")\n\n        missing_files = [str(r["path"]) for r in img_refs if not r["exists"]] + [\n            str(a["path"]) for a in audio_refs if not a["exists"]\n        ]\n        jobs.append({\n            "part": tag,\n            "episode": args.episode,\n            "shots_count": len(shots),\n            "duration_sec": seg_dur,\n            "frame_length": frame_length,\n            "actual_duration_sec": round(frame_length / FPS, 2),\n            "width": width,\n            "height": height,\n            "fps": FPS,\n            "seed": seed,\n            "ref_image_size": ref_image_size,\n            "use_turbo": args.turbo,\n            "ref_images": [\n                {"tag": r["tag"], "kind": r["kind"], "name": r["name"], "src_rel": os.path.relpath(r["path"], series_dir).replace("\\\\", "/"), "staged_name": staged_imgs[i], "exists": r["exists"]}\n                for i, r in enumerate(img_refs)\n            ],\n            "ref_audios": [\n                {"tag": a["tag"], "name": a["name"], "src_rel": os.path.relpath(a["path"], series_dir).replace("\\\\", "/"), "staged_name": staged_auds[i], "exists": a["exists"]}\n                for i, a in enumerate(audio_refs)\n            ],\n            "missing_files": missing_files,\n            "prompt_file": str(prompt_file.name),\n            "api_file": str(api_file.name),\n            "output_prefix": output_prefix,\n            "target_clip_rel": f"{args.episode}/clips/{args.episode}{part_suffix}_h3.mp4",\n        })\n\n    manifest = {\n        "schema": "ai-anime-h3/v1",\n        "series_title": s.get("title", series_dir.name),\n        "episode": args.episode,\n        "episode_title": e.get("title", args.episode),\n        "profile_name": args.profile,\n        "profile": profile,\n        "use_turbo": args.turbo,\n        "turbo_lora": TURBO_LORA if args.turbo else None,\n        "resolution": args.resolution,\n        "jobs": jobs,\n    }\n    manifest_path = out_dir / "manifest.json"\n    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")\n\n    print(f"✅ 已生成 MiniMax-H3 任务清单 ({args.profile}, ref_size={ref_image_size}): {manifest_path}")\n    for j in jobs:\n        tags = " / ".join(f"{r[\'tag\']}:{r[\'name\']}" for r in j["ref_images"])\n        auds = " / ".join(f"{a[\'tag\']}:{a[\'name\']}" for a in j["ref_audios"])\n        aud_str = f" | 音频: {auds}" if auds else ""\n        print(f"  - 段 {j[\'part\']} ({j[\'shots_count\']}格): {j[\'width\']}x{j[\'height\']} @ 24fps, {j[\'frame_length\']}帧 (~{j[\'actual_duration_sec\']}s) | {tags}{aud_str}")\n        if j["missing_files"]:\n            print(f"    ⚠️ 尚未放图的素材: {j[\'missing_files\']}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
Path('/content/colab_h3_worker.py').write_text('# -*- coding: utf-8 -*-\n"""h3出片 Colab 云端执行引擎（支持 G4 96GB / H200 141GB / H100 80GB）。\n\n核心能力：\n  1) 打印硬件与磁盘状态，防爆盘硬检查；\n  2) 自动安装/更新 ComfyUI + sageattention + hf_transfer；\n  3) 按 manifest.json 精确拉取所需权重（零双份占盘）：\n     - 优先命中 Google Drive rclone 精确单文件缓存（Google 内网极速直拉原位）；\n     - 未命中则免密（或可选 HF_TOKEN）走 hf_transfer 直连 Comfy-Org/MiniMax-H3 下载，下完顺手回写 G 盘缓存复用；\n     - 若传入本地挂载目录 --model-cache-dir，同盘自动软链、异盘单文件拷贝；\n  4) 启动 ComfyUI（默认动态卸载管理，G4 96GB + 176GB RAM 稳跑无损 BF16；自动启用 --use-sage-attention）；\n  5) batch 模式：逐段加载参考图/音频/故事板 -> 采样 -> 音画联合解码 -> ffprobe 校验 -> 写入 epNN/clips/；\n  6) webui 模式：开启 Cloudflare Tunnel 输出 ComfyUI 网页地址供手动精调。\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\nfrom typing import Any\n\nCOMFY_DIR = Path("/content/ComfyUI")\nREPO_ID = "Comfy-Org/MiniMax-H3"\n\n\ndef run_cmd(cmd: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:\n    print(f"$ {\' \'.join(cmd)}", flush=True)\n    proc = subprocess.run(cmd, cwd=str(cwd) if cwd else None)\n    if check and proc.returncode != 0:\n        raise RuntimeError(f"命令失败 (rc={proc.returncode}): {\' \'.join(cmd)}")\n    return proc\n\n\ndef print_system_banner() -> None:\n    print("=== Colab 云端硬件与磁盘状态 ===", flush=True)\n    subprocess.run(\n        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],\n        check=False,\n    )\n    subprocess.run(["free", "-h"], check=False)\n    subprocess.run(["df", "-h", "/content"], check=False)\n    print("================================", flush=True)\n\n\ndef free_gb(path: Path = Path("/content")) -> float:\n    return shutil.disk_usage(str(path)).free / (1024**3)\n\n\ndef ensure_comfyui(install_tunnel: bool = False) -> None:\n    if not COMFY_DIR.exists():\n        run_cmd(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI", str(COMFY_DIR)])\n    else:\n        run_cmd(["git", "-C", str(COMFY_DIR), "pull", "--ff-only"], check=False)\n\n    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY_DIR / "requirements.txt")])\n    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf_xet"])\n    # SageAttention 在 G4 (Blackwell) / H200 / H100 上可提供显著加速；若无预编译轮子则平滑回退原生 Flash/SDPA\n    sage = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sageattention"])\n    if sage.returncode != 0:\n        print("⚠️ sageattention 安装失败，将回退至 PyTorch 原生 Flash/SDPA 注意力。", flush=True)\n\n    if install_tunnel and not Path("/usr/local/bin/cloudflared").exists():\n        run_cmd([\n            "wget", "-q",\n            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",\n            "-O", "/usr/local/bin/cloudflared",\n        ])\n        run_cmd(["chmod", "+x", "/usr/local/bin/cloudflared"])\n\n\ndef rclone_remote_exists(rclone_conf: str, remote_file: str) -> bool:\n    proc = subprocess.run(\n        ["rclone", "--config", rclone_conf, "lsf", remote_file],\n        capture_output=True, text=True,\n    )\n    return proc.returncode == 0 and bool(proc.stdout.strip())\n\n\ndef _dir_bytes(root: Path) -> int:\n    total = 0\n    if not root.exists():\n        return 0\n    for p in root.rglob("*"):\n        try:\n            if p.is_file():\n                total += p.stat().st_size\n        except OSError:\n            pass\n    return total\n\n\ndef _copy_with_heartbeat(src: Path, dst: Path, label: str) -> None:\n    total = src.stat().st_size\n    done = 0\n    t0 = time.time()\n    last = t0\n    tmp_dst = dst.with_suffix(dst.suffix + ".part")\n    with src.open("rb") as f_in, tmp_dst.open("wb") as f_out:\n        while True:\n            buf = f_in.read(16 * 1024 * 1024)\n            if not buf:\n                break\n            f_out.write(buf)\n            done += len(buf)\n            now = time.time()\n            if now - last >= 15.0:\n                pct = (done / total * 100.0) if total else 0.0\n                spd = done / max(0.1, now - t0) / 1e6\n                print(\n                    f"   ⏳ 拷贝缓存 [{label}]: {done / 1e9:.2f}/{total / 1e9:.2f} GB ({pct:.1f}%) | ~{spd:.0f} MB/s",\n                    flush=True,\n                )\n                last = now\n    tmp_dst.replace(dst)\n    elapsed = max(1, int(time.time() - t0))\n    print(f"✅ 挂载缓存就绪: {label} ({total / 1e9:.2f} GB, 耗时 {elapsed}s, ~{total / elapsed / 1e6:.0f} MB/s)", flush=True)\n\n\ndef _download_with_heartbeat(repo_id: str, repo_path: str, local_dir: Path, token: str | bool) -> None:\n    from huggingface_hub import hf_hub_download\n\n    stop_evt = threading.Event()\n    t0 = time.time()\n    base_bytes = _dir_bytes(local_dir)\n\n    def _beat() -> None:\n        while not stop_evt.wait(15.0):\n            delta = max(0, _dir_bytes(local_dir) - base_bytes)\n            print(f"   ⏳ 下载中 [{repo_path}]: +{delta / 1e9:.2f} GB | 已耗时 {int(time.time() - t0)}s", flush=True)\n\n    th = threading.Thread(target=_beat, daemon=True)\n    th.start()\n    try:\n        hf_hub_download(\n            repo_id=repo_id,\n            filename=repo_path,\n            local_dir=str(local_dir),\n            token=token,\n        )\n    finally:\n        stop_evt.set()\n        th.join(timeout=2.0)\n\n\ndef ensure_models(\n    manifest: dict[str, Any],\n    hf_token: str | None,\n    model_cache_dir: Path | None,\n    rclone_model_cache: str | None,\n    rclone_conf: str | None,\n    no_cache_push: bool = False,\n) -> None:\n    os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"\n\n    prof = manifest["profile"]\n    needed: list[tuple[str, str]] = [\n        ("diffusion_models", prof["unet"]),\n        ("text_encoders", prof["clip"]),\n        ("vae", prof["video_vae"]),\n        ("vae", prof["audio_vae"]),\n    ]\n    if manifest.get("use_turbo") and manifest.get("turbo_lora"):\n        needed.append(("loras", manifest["turbo_lora"]))\n\n    models_root = COMFY_DIR / "models"\n    models_root.mkdir(parents=True, exist_ok=True)\n\n    for sub, fname in needed:\n        dst = models_root / sub / fname\n        dst.parent.mkdir(parents=True, exist_ok=True)\n        if dst.exists() and dst.stat().st_size > 1_000_000:\n            print(f"✅ 已有本地权重: {sub}/{fname} ({dst.stat().st_size / 1e9:.2f} GB)", flush=True)\n            continue\n\n        avail = free_gb(Path("/content"))\n        if avail < 12.0:\n            raise RuntimeError(f"容器 /content 可用磁盘仅剩 {avail:.1f} GB，无法继续拉取 {sub}/{fname}！")\n\n        # 通道 1：rclone 精确单文件内网缓存（零双份占盘）\n        if rclone_model_cache and rclone_conf and Path(rclone_conf).exists():\n            remote_file = f"{rclone_model_cache.rstrip(\'/\')}/{sub}/{fname}"\n            if rclone_remote_exists(rclone_conf, remote_file):\n                print(f"⚡ 命中 Google Drive 缓存，内网直拉原位: {remote_file} -> {dst}", flush=True)\n                run_cmd([\n                    "rclone", "--config", rclone_conf, "copyto",\n                    remote_file, str(dst),\n                    "--drive-chunk-size", "128M",\n                    "--stats", "15s", "--stats-one-line",\n                ])\n                if dst.exists() and dst.stat().st_size > 1_000_000:\n                    print(f"✅ 内网缓存就绪: {sub}/{fname} ({dst.stat().st_size / 1e9:.2f} GB)", flush=True)\n                    continue\n\n        # 通道 2：本地/Drive FUSE 目录精确单文件缓存（同盘软链防双份，异盘流式单文件拷贝带进度心跳）\n        if model_cache_dir is not None:\n            cached = model_cache_dir / sub / fname\n            if cached.exists() and cached.stat().st_size > 1_000_000:\n                same_dev = os.stat(cached.parent).st_dev == os.stat(dst.parent).st_dev\n                if same_dev:\n                    print(f"🔗 同盘零拷贝软链缓存: {cached} -> {dst}", flush=True)\n                    os.symlink(cached, dst)\n                else:\n                    print(f"📦 从 Google Drive 挂载缓存流式拷贝到高速 SSD: {cached} -> {dst}", flush=True)\n                    _copy_with_heartbeat(cached, dst, f"{sub}/{fname}")\n                continue\n\n        # 通道 3：从公开仓库 Comfy-Org/MiniMax-H3 直拉（无 Token 显式传 False 跳过 Colab vault 超时等待）\n        repo_path = f"{sub}/{fname}"\n        tok: str | bool = (hf_token or os.environ.get("HF_TOKEN") or "").strip() or False\n        print(f"⬇️ 从 Hugging Face 拉取 {REPO_ID}/{repo_path} (余盘 {free_gb():.1f} GB) ...", flush=True)\n        _download_with_heartbeat(REPO_ID, repo_path, models_root, tok)\n        # 清理 .cache 临时元数据碎片，节省磁盘\n        dl_cache = models_root / ".cache"\n        if dl_cache.exists():\n            shutil.rmtree(dl_cache, ignore_errors=True)\n\n        if not dst.exists() or dst.stat().st_size < 1_000_000:\n            raise RuntimeError(f"模型下载不完整: {dst}")\n        print(f"✅ HF 下载完成: {sub}/{fname} ({dst.stat().st_size / 1e9:.2f} GB，余盘 {free_gb():.1f} GB)", flush=True)\n\n        # 回写持久化缓存供后续秒拉\n        if not no_cache_push:\n            if rclone_model_cache and rclone_conf and Path(rclone_conf).exists():\n                remote_file = f"{rclone_model_cache.rstrip(\'/\')}/{sub}/{fname}"\n                print(f"💾 同步备份权重至 Google Drive 缓存: {remote_file}", flush=True)\n                run_cmd([\n                    "rclone", "--config", rclone_conf, "copyto",\n                    str(dst), remote_file,\n                    "--drive-chunk-size", "128M",\n                    "--stats", "15s", "--stats-one-line",\n                ], check=False)\n            elif model_cache_dir is not None:\n                cache_dst = model_cache_dir / sub / fname\n                cache_dst.parent.mkdir(parents=True, exist_ok=True)\n                if not cache_dst.exists() and os.stat(cache_dst.parent).st_dev != os.stat(dst.parent).st_dev:\n                    print(f"💾 同步备份权重至挂载缓存: {cache_dst}", flush=True)\n                    shutil.copy2(dst, cache_dst)\n\n\ndef http_json(url: str, payload: dict[str, Any] | None = None, timeout: int = 30) -> Any:\n    data = json.dumps(payload).encode("utf-8") if payload is not None else None\n    headers = {"Content-Type": "application/json"} if data is not None else {}\n    req = urllib.request.Request(url, data=data, headers=headers)\n    with urllib.request.urlopen(req, timeout=timeout) as resp:\n        return json.loads(resp.read().decode("utf-8"))\n\n\ndef has_sageattention() -> bool:\n    try:\n        import sageattention  # noqa: F401\n        return True\n    except Exception:\n        return False\n\n\ndef tail_log(log_path: Path, lines: int = 80) -> str:\n    if not log_path.exists():\n        return "(无日志文件)"\n    content = log_path.read_text(encoding="utf-8", errors="replace").splitlines()\n    return "\\n".join(content[-lines:])\n\n\ndef start_comfy_server(port: int = 8188, listen: str = "127.0.0.1") -> tuple[subprocess.Popen, Path]:\n    cmd = [sys.executable, str(COMFY_DIR / "main.py"), "--listen", listen, "--port", str(port)]\n    if has_sageattention():\n        cmd.append("--use-sage-attention")\n        print("⚡ 已启用 --use-sage-attention", flush=True)\n    log_path = Path("/tmp/comfyui_h3.log")\n    log_f = log_path.open("w", encoding="utf-8")\n    proc = subprocess.Popen(cmd, cwd=str(COMFY_DIR), stdout=log_f, stderr=subprocess.STDOUT)\n\n    base = f"http://127.0.0.1:{port}"\n    deadline = time.time() + 180\n    while time.time() < deadline:\n        if proc.poll() is not None:\n            raise RuntimeError(f"ComfyUI 启动失败 (rc={proc.returncode}):\\n{tail_log(log_path)}")\n        try:\n            info = http_json(f"{base}/object_info/MiniMaxH3ReferenceToVideo", timeout=5)\n            if "MiniMaxH3ReferenceToVideo" in info:\n                print("✅ ComfyUI 服务就绪，已确认原生节点 MiniMaxH3ReferenceToVideo", flush=True)\n                return proc, log_path\n        except Exception:\n            time.sleep(2)\n    proc.terminate()\n    raise RuntimeError(f"ComfyUI 启动超时或缺少 MiniMaxH3ReferenceToVideo 节点:\\n{tail_log(log_path)}")\n\n\ndef patch_save_video_node(graph: dict[str, Any], base_url: str) -> None:\n    """根据当前 ComfyUI 的 SaveVideo schema 自动适配 format/codec 入参。"""\n    try:\n        schema = http_json(f"{base_url}/object_info/SaveVideo", timeout=10)\n        req_inputs = schema.get("SaveVideo", {}).get("input", {}).get("required", {})\n        fmt_spec = req_inputs.get("format")\n        if isinstance(fmt_spec, list) and fmt_spec and isinstance(fmt_spec[0], list):\n            allowed_fmt = fmt_spec[0]\n            if graph["39"]["inputs"].get("format") not in allowed_fmt:\n                graph["39"]["inputs"]["format"] = "mp4" if "mp4" in allowed_fmt else ("auto" if "auto" in allowed_fmt else allowed_fmt[0])\n        if "codec" in req_inputs:\n            codec_spec = req_inputs["codec"]\n            if isinstance(codec_spec, list) and codec_spec and isinstance(codec_spec[0], list):\n                allowed_codec = codec_spec[0]\n                graph["39"]["inputs"]["codec"] = "auto" if "auto" in allowed_codec else allowed_codec[0]\n            else:\n                graph["39"]["inputs"]["codec"] = "auto"\n    except Exception:\n        pass\n\n\ndef stage_job_assets(series_dir: Path, job: dict[str, Any]) -> None:\n    input_dir = COMFY_DIR / "input"\n    input_dir.mkdir(parents=True, exist_ok=True)\n    for item in job["ref_images"]:\n        src_rel = item["src_rel"].replace("\\\\", "/")\n        src = series_dir / src_rel\n        if not src.exists():\n            raise FileNotFoundError(f"缺少参考图 ({item[\'tag\']} {item[\'name\']}): {src}")\n        shutil.copy2(src, input_dir / item["staged_name"])\n    for item in job["ref_audios"]:\n        src_rel = item["src_rel"].replace("\\\\", "/")\n        src = series_dir / src_rel\n        if not src.exists():\n            raise FileNotFoundError(f"缺少参考音频 ({item[\'tag\']} {item[\'name\']}): {src}")\n        shutil.copy2(src, input_dir / item["staged_name"])\n\n\ndef probe_video(path: Path) -> dict[str, Any]:\n    r = subprocess.run(\n        [\n            "ffprobe", "-v", "error",\n            "-show_entries", "format=duration:stream=codec_type,codec_name,width,height,r_frame_rate,sample_rate,channels",\n            "-of", "json", str(path),\n        ],\n        capture_output=True, text=True, encoding="utf-8", errors="replace",\n    )\n    if r.returncode != 0:\n        raise RuntimeError(f"ffprobe 失败 ({path}): {r.stderr}")\n    info = json.loads(r.stdout or "{}")\n    v = next((s for s in info.get("streams", []) if s.get("codec_type") == "video"), None)\n    a = next((s for s in info.get("streams", []) if s.get("codec_type") == "audio"), None)\n    dur = float(info.get("format", {}).get("duration", 0.0))\n    if not v or dur <= 0.5:\n        raise RuntimeError(f"输出视频校验异常: {path} (duration={dur}, video={v})")\n    return {\n        "duration": round(dur, 2),\n        "width": v.get("width"),\n        "height": v.get("height"),\n        "fps": v.get("r_frame_rate"),\n        "vcodec": v.get("codec_name"),\n        "has_audio": a is not None,\n        "acodec": a.get("codec_name") if a else None,\n        "sample_rate": a.get("sample_rate") if a else None,\n        "channels": a.get("channels") if a else None,\n    }\n\n\ndef run_batch(\n    series_dir: Path,\n    episode: str,\n    manifest: dict[str, Any],\n    base_url: str,\n    server: subprocess.Popen,\n    log_path: Path,\n    skip_existing: bool = False,\n    only_parts: set[str] | None = None,\n    rclone_clips_dest: str | None = None,\n    rclone_conf: str | None = None,\n    local_clips_dest: Path | None = None,\n) -> list[Path]:\n    ep_dir = series_dir / episode\n    jobs_dir = ep_dir / "h3_jobs"\n    output_dir = COMFY_DIR / "output"\n    produced: list[Path] = []\n\n    selected_jobs = [j for j in manifest["jobs"] if (not only_parts or j["part"] in only_parts)]\n    if only_parts:\n        missing_parts = sorted(only_parts - {j["part"] for j in selected_jobs})\n        if missing_parts:\n            raise ValueError(f"--only-parts 指定了 manifest 中不存在的段号: {missing_parts}")\n        print(f"🎯 按 --only-parts 筛选执行 {len(selected_jobs)}/{len(manifest[\'jobs\'])} 段: {[j[\'part\'] for j in selected_jobs]}", flush=True)\n\n    for job in selected_jobs:\n        part = job["part"]\n        target = series_dir / job["target_clip_rel"].replace("\\\\", "/")\n        if skip_existing and target.exists() and target.stat().st_size > 100_000:\n            print(f"⏭️ 跳过已有成片段 {part}: {target}", flush=True)\n            produced.append(target)\n            continue\n\n        print(\n            f"\\n🎬 开始生成 {episode} 段 {part} ({job[\'width\']}x{job[\'height\']}, "\n            f"{job[\'frame_length\']}帧 ~{job[\'actual_duration_sec\']}s, ref_size={job[\'ref_image_size\']}, seed={job[\'seed\']}) ...",\n            flush=True,\n        )\n        stage_job_assets(series_dir, job)\n\n        graph = json.loads((jobs_dir / job["api_file"]).read_text(encoding="utf-8"))\n        patch_save_video_node(graph, base_url)\n\n        t0 = time.time()\n        try:\n            resp = http_json(f"{base_url}/prompt", payload={"prompt": graph}, timeout=30)\n        except urllib.error.HTTPError as exc:\n            body = exc.read().decode("utf-8", errors="replace")\n            raise RuntimeError(f"提交 ComfyUI 任务被拒 (HTTP {exc.code}): {body}\\n--- ComfyUI 日志 ---\\n{tail_log(log_path)}")\n        prompt_id = resp.get("prompt_id")\n        if not prompt_id:\n            raise RuntimeError(f"提交 ComfyUI 任务失败: {resp}")\n        print(f"   prompt_id={prompt_id}，等待 Qwen3-VL 编码、DiT 采样与音画解码...", flush=True)\n\n        out_video: Path | None = None\n        last_beat = time.time()\n        while True:\n            time.sleep(5)\n            if server.poll() is not None:\n                raise RuntimeError(f"ComfyUI 进程异常退出 (rc={server.returncode}):\\n{tail_log(log_path)}")\n            hist = http_json(f"{base_url}/history/{prompt_id}", timeout=15)\n            if prompt_id in hist:\n                entry = hist[prompt_id]\n                status = entry.get("status", {})\n                if status.get("status_str") == "error":\n                    raise RuntimeError(\n                        f"ComfyUI 执行报错 ({prompt_id}): {json.dumps(status, ensure_ascii=False)}\\n"\n                        f"--- ComfyUI 日志 ---\\n{tail_log(log_path)}"\n                    )\n                outputs = entry.get("outputs", {})\n                for node_out in outputs.values():\n                    for key in ("videos", "images", "gifs"):\n                        for item in node_out.get(key, []):\n                            fname = item.get("filename", "")\n                            if fname.lower().endswith((".mp4", ".webm", ".mkv", ".mov")):\n                                sub = item.get("subfolder", "")\n                                cand = output_dir / sub / fname if sub else output_dir / fname\n                                if cand.exists():\n                                    out_video = cand\n                                    break\n                break\n            if time.time() - last_beat >= 20:\n                print(f"   ⏳ 段 {part} 运行中 ({int(time.time() - t0)}s) ...", flush=True)\n                last_beat = time.time()\n\n        if out_video is None:\n            cands = sorted(output_dir.rglob(f"{job[\'output_prefix\']}*.mp4"), key=lambda p: p.stat().st_mtime, reverse=True)\n            if not cands:\n                raise RuntimeError(f"未在 {output_dir} 找到前缀 {job[\'output_prefix\']} 的输出视频\\n{tail_log(log_path)}")\n            out_video = cands[0]\n\n        meta = probe_video(out_video)\n        target.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copy2(out_video, target)\n        elapsed = int(time.time() - t0)\n        aud_info = f"{meta[\'acodec\']} {meta[\'sample_rate\']}Hz {meta[\'channels\']}ch" if meta["has_audio"] else "无音轨!"\n        print(\n            f"✅ 段 {part} 完成 ({elapsed}s): {target} | "\n            f"{meta[\'width\']}x{meta[\'height\']} {meta[\'fps\']}fps ({meta[\'vcodec\']}), {meta[\'duration\']}s, 音频={aud_info}",\n            flush=True,\n        )\n        if not meta["has_audio"]:\n            print("⚠️ 警告：输出视频未检测到音轨，请检查 ComfyUI 日志与音频 VAE 配置！", flush=True)\n        if rclone_clips_dest and rclone_conf and Path(rclone_conf).exists():\n            remote_clip = f"{rclone_clips_dest.rstrip(\'/\')}/{target.name}"\n            print(f"☁️ 实时备份成片至 Google Drive: {remote_clip}", flush=True)\n            run_cmd([\n                "rclone", "--config", rclone_conf, "copyto",\n                str(target), remote_clip,\n                "--drive-chunk-size", "64M",\n            ], check=False)\n        if local_clips_dest is not None:\n            local_clips_dest.mkdir(parents=True, exist_ok=True)\n            dst_mp4 = local_clips_dest / target.name\n            shutil.copy2(target, dst_mp4)\n            if hasattr(os, "sync"):\n                os.sync()\n            print(f"💾 实时落盘挂载云盘: {dst_mp4}", flush=True)\n        produced.append(target)\n\n    return produced\n\n\ndef run_webui(port: int = 8188) -> None:\n    def _tunnel():\n        p = subprocess.Popen(\n            ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],\n            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,\n        )\n        assert p.stdout is not None\n        for line in p.stdout:\n            m = re.search(r"https://[a-zA-Z0-9-]+\\.trycloudflare\\.com", line)\n            if m:\n                print(f"\\n🚀 ComfyUI 公网链接: {m.group(0)}\\n", flush=True)\n\n    threading.Thread(target=_tunnel, daemon=True).start()\n    print("🌐 WebUI 模式运行中（在 Colab 中点击停止按钮即可结束）...", flush=True)\n    while True:\n        time.sleep(30)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description="h3出片 Colab MiniMax-H3 云端执行引擎")\n    parser.add_argument("--series-dir", required=True, help="系列目录（容器内路径）")\n    parser.add_argument("--episode", required=True, help="集目录名（如 ep01）")\n    parser.add_argument("--mode", choices=["batch", "webui"], default="batch")\n    parser.add_argument("--hf-token", default=None, help="可选 Hugging Face Token（公开仓库可不传）")\n    parser.add_argument("--model-cache-dir", default=None, help="可选本地/挂载模型缓存目录")\n    parser.add_argument("--rclone-model-cache", default=None, help="可选 rclone 远程缓存路径（如 gdrive:models/minimax_h3）")\n    parser.add_argument("--rclone-config", default="/tmp/rclone.conf", help="rclone 配置文件路径")\n    parser.add_argument("--no-cache-push", action="store_true", help="下载新权重后不回传至远程缓存")\n    parser.add_argument("--skip-existing", action="store_true", help="跳过目标 clips 目录中已存在的成片")\n    parser.add_argument("--only-parts", default="", help="仅运行指定段号（逗号分隔，如 S01 或 S02,S03,S04）")\n    parser.add_argument("--rclone-clips-dest", default=None, help="每完成一段即刻回传至 Google Drive 远端 clips 目录")\n    parser.add_argument("--local-clips-dest", default=None, help="每完成一段即刻拷贝至本地/挂载云盘 clips 目录并刷盘")\n    parser.add_argument("--port", type=int, default=8188)\n    args = parser.parse_args()\n\n    print_system_banner()\n\n    series_dir = Path(args.series_dir).resolve()\n    manifest_path = series_dir / args.episode / "h3_jobs" / "manifest.json"\n    if not manifest_path.exists():\n        raise FileNotFoundError(f"未找到任务清单，请先运行 pack_direct_job.py 或 make_h3_jobs.py: {manifest_path}")\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n    print(f"📋 加载任务清单: {manifest.get(\'series_title\')} / {args.episode} | profile={manifest.get(\'profile_name\')} | 共 {len(manifest.get(\'jobs\', []))} 段", flush=True)\n\n    ensure_comfyui(install_tunnel=(args.mode == "webui"))\n    cache_dir = Path(args.model_cache_dir) if args.model_cache_dir else None\n    ensure_models(\n        manifest=manifest,\n        hf_token=args.hf_token,\n        model_cache_dir=cache_dir,\n        rclone_model_cache=args.rclone_model_cache,\n        rclone_conf=args.rclone_config,\n        no_cache_push=args.no_cache_push,\n    )\n\n    listen = "0.0.0.0" if args.mode == "webui" else "127.0.0.1"\n    server, log_path = start_comfy_server(port=args.port, listen=listen)\n    try:\n        if args.mode == "webui":\n            if manifest["jobs"]:\n                try:\n                    stage_job_assets(series_dir, manifest["jobs"][0])\n                    print(f"📂 已将参考素材预置到 {COMFY_DIR / \'input\'}", flush=True)\n                except FileNotFoundError as exc:\n                    print(f"⚠️ 预置参考素材提醒: {exc}", flush=True)\n            run_webui(port=args.port)\n        else:\n            only_set = {p.strip() for p in args.only_parts.split(",") if p.strip()} or None\n            local_dest = Path(args.local_clips_dest) if args.local_clips_dest else None\n            produced = run_batch(\n                series_dir=series_dir,\n                episode=args.episode,\n                manifest=manifest,\n                base_url=f"http://127.0.0.1:{args.port}",\n                server=server,\n                log_path=log_path,\n                skip_existing=args.skip_existing,\n                only_parts=only_set,\n                rclone_clips_dest=args.rclone_clips_dest,\n                rclone_conf=args.rclone_config,\n                local_clips_dest=local_dest,\n            )\n            print(f"\\n🎉 全部 {len(produced)} 个片段生成并校验完成！", flush=True)\n    finally:\n        server.terminate()\n        try:\n            server.wait(timeout=10)\n        except Exception:\n            server.kill()\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')

LOCAL_SERIES = Path('/content/series_work')
if LOCAL_SERIES.exists():
    shutil.rmtree(LOCAL_SERIES)
shutil.copytree(SERIES_DRIVE_DIR, LOCAL_SERIES)
print('✅ 已同步系列目录至本地高速盘:', LOCAL_SERIES)


In [ ]:
#@title 3. 检查本次待跑段落的 H3 任务清单与提示词
import subprocess, sys, json
from pathlib import Path

manifest_path = Path('/content/series_work') / EPISODE / 'h3_jobs' / 'manifest.json'
if not (USE_EXISTING_MANIFEST and manifest_path.exists()):
    cmd = [
        sys.executable, '/content/make_h3_jobs.py',
        '/content/series_work', EPISODE,
        '--profile', PROFILE,
        '--resolution', RESOLUTION,
        '--ref-size', REF_IMAGE_SIZE,
        '--use-custom-prompt',
    ]
    if ENABLE_TURBO_4STEP:
        cmd.append('--turbo')
    subprocess.run(cmd, check=True)
else:
    print('✅ 直接沿用本地编译好的 manifest.json:', manifest_path)

manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
only_set = {p.strip() for p in ONLY_PARTS.split(',') if p.strip()}
selected = [j for j in manifest['jobs'] if (not only_set or j['part'] in only_set)]
assert selected, f'按 ONLY_PARTS={ONLY_PARTS!r} 未匹配到段落！可选: {[j["part"] for j in manifest["jobs"]]}'
print(f'📋 本次将执行 {len(selected)}/{len(manifest["jobs"])} 段: {[j["part"] for j in selected]}')
for j in selected:
    tags = ' / '.join(f"{r['tag']}:{r['name']}" for r in j['ref_images'])
    print(f"  - 段 {j['part']}: {j['width']}x{j['height']} @ {j['fps']}fps, {j['frame_length']}帧 (~{j['actual_duration_sec']}s) | {tags}")
    if RUN_MODE == 'batch' and j.get('missing_files'):
        raise FileNotFoundError(f'缺少参考素材，请补齐后重试: {j["missing_files"]}')
if len(selected) <= 2:
    for j in selected:
        p_txt = (manifest_path.parent / j['prompt_file']).read_text(encoding='utf-8')
        print(f'\n--- 段 {j["part"]} H3 提示词全文 ---\n{p_txt}')


In [ ]:
#@title 4. 启动 MiniMax-H3 无损生成（实时日志流 + 每段即刻落盘 G 盘 + 自动关机）
import os, subprocess, sys, shutil, time
from pathlib import Path
from google.colab import runtime

drive_clips_dir = Path(SERIES_DRIVE_DIR) / EPISODE / 'clips'
drive_clips_dir.mkdir(parents=True, exist_ok=True)

worker_cmd = [
    sys.executable, '-u', '/content/colab_h3_worker.py',
    '--series-dir', '/content/series_work',
    '--episode', EPISODE,
    '--mode', RUN_MODE,
    '--local-clips-dest', str(drive_clips_dir),
]
if MODEL_CACHE_DRIVE_DIR.strip():
    worker_cmd += ['--model-cache-dir', MODEL_CACHE_DRIVE_DIR.strip()]
if ONLY_PARTS.strip():
    worker_cmd += ['--only-parts', ONLY_PARTS.strip()]
if SKIP_EXISTING:
    worker_cmd.append('--skip-existing')

env = dict(os.environ)
env['PYTHONUNBUFFERED'] = '1'
print('🚀 启动命令:', ' '.join(worker_cmd), flush=True)

ok = False
try:
    proc = subprocess.Popen(
        worker_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='', flush=True)
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'colab_h3_worker.py 失败 (rc={rc})')
    ok = True
    if RUN_MODE == 'batch':
        src_clips = Path('/content/series_work') / EPISODE / 'clips'
        for mp4 in sorted(src_clips.glob('*.mp4')):
            dst_mp4 = drive_clips_dir / mp4.name
            if not dst_mp4.exists() or dst_mp4.stat().st_size != mp4.stat().st_size:
                shutil.copy2(mp4, dst_mp4)
            print(f'💾 已核验写入 Google Drive: {dst_mp4} ({dst_mp4.stat().st_size / 1e6:.2f} MB)', flush=True)
        os.sync()
        print('⏳ 等待 10 秒确保 Google Drive 挂载缓存彻底刷盘...', flush=True)
        time.sleep(10)
        print('🎉 本批成片已安全落盘 Google Drive！', flush=True)
finally:
    if RUN_MODE == 'batch' and AUTO_UNASSIGN_AFTER_BATCH:
        if not ok:
            print('⚠️ 任务报错，为防持续扣费仍将在 5 秒后自动释放 GPU（请先截图或查看上方报错日志）...', flush=True)
            time.sleep(5)
        print('🛑 正在自动释放 GPU 运行时以停止计费...', flush=True)
        runtime.unassign()


In [ ]:
#@title 5. 手动立刻释放 GPU（仅调试或 WebUI 模式需要时点此停止计费）
from google.colab import runtime
runtime.unassign()
